# Reproducing Gu et al. (2026) from dandiset 001836

Paper. Gu S, Liu C, Gillespie AK, Nevers R, Denovellis EL, Coulter ME, Kay K, Frank LM (2026).
Naturalistic behavior and self-generated neural activity predictive of self-correction. bioRxiv,
[doi:10.64898/2026.05.26.727951](https://doi.org/10.64898/2026.05.26.727951). The claims file
(`claims/001836/10.64898-2026.05.26.727951.yaml`) was written from the bioRxiv preprint, version 1
(May 27, 2026), with its supplement (Materials and Methods, Figures S1 to S12).

Dandiset. [DANDI:001836](https://dandiarchive.org/dandiset/001836/0.261030.0428), version
0.261030.0428. The paper cites the dandiset without a version. Two versions are published,
0.260526.1804 and 0.261030.0428; the cell below shows that they hold the same 2494 assets with the
same blobs, so they differ only in metadata. The most recent one is pinned, as the protocol asks
when the paper cites none.

What is tested. The dandiset holds the raw recordings (30 kHz voltage, inertial sensors, LED
tracking, well pokes and pumps, Statescript logs), a 1 kHz LFP file per day, and Spyglass analysis
files: head position, linearized position, and decodes triggered on events. The `scratch/triggered`
tables carry no description; they are identified here by aligning their trigger times with this
entry's COM times and outer pokes, and read with the paper's code (Gu2026 at commit 0f0c9a7), which
wrote them. They hold the decoder's MAP position in windows around COM times and outer pokes, which
is what the Figure 3 claims use. The dandiset does not hold the decoder's posterior over whole
sessions, its state probabilities, the remote or local extended events, theta cycles or ripples.
The behavioral claims (Figure 1, Figure S1F) and the Figure 3 claims are tested, and so is the SWR
claim (Figure S7B), with ripples detected on the deposited LFP. The other claims on hippocampal
content need the decoder output over every COM event, which would have to be recomputed from the
raw voltage (12.5 TB of raw files), and are blocked.

| Claim | Status |
|---|---|
| fig1c-learning | tested |
| fig1e-trial-counts | tested |
| fig1d-initial-switch-bias | tested |
| fig1d-switch-bias-declines | tested |
| figs1f-multiple-com | tested |
| fig1g-com-corrective | tested |
| fig1g-com-matches-run-through | tested |
| fig1h-final-choice-less-switching | tested |
| fig1h-final-ratio-near-one | tested |
| fig3b-extended-before-stopping | tested |
| fig3b-distance-predicts-com | tested |
| fig3c-distant-after-stopping | tested |
| fig3d-distant-vs-inbound | tested |
| figs7b-no-swr | tested |
| figs6, figs3, figs11b, fig4a (two), fig4b, figs11a, fig4c, fig4d, fig4e, figs11c, fig5a, fig5b, figs8, figs10 | blocked (too_large) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

import comtask as ct

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = ct.list_assets()
print(f"dandiset {ct.DANDISET_ID} version {ct.DANDISET_VERSION}: {len(assets)} assets, "
      f"{assets['size'].sum() / 1e12:.2f} TB")
R = ct.load_all(assets)
print(f"loaded {len(R)} rat-days in {time.time() - started:.0f} s")

## Data check

In [ ]:
# The two published versions: same paths and blobs?
other = ct._get(f"{ct.API}/dandisets/{ct.DANDISET_ID}/versions/")
versions = [v["version"] for v in other["results"] if v["version"] != "draft"]
print("published versions:", versions)
for v in versions:
    if v == ct.DANDISET_VERSION:
        continue
    url = f"{ct.API}/dandisets/{ct.DANDISET_ID}/versions/{v}/assets/?page_size=1000"
    blobs = {}
    while url:
        page = ct._get(url)
        blobs.update({a["path"]: a["blob"] for a in page["results"]})
        url = page["next"]
    same = sum(blobs.get(p) == b for p, b in zip(assets.path, assets.blob))
    print(f"version {v}: {len(blobs)} assets, {same} with the same path and blob as {ct.DANDISET_VERSION}")

print("\nAssets per rat and kind:")
print(pd.crosstab(assets.rat, assets.kind).to_string())
print("\nDays per rat:", assets.groupby("rat").day.nunique().to_dict())
kinds = pd.DataFrame({d: r["obj_counts"] for d, r in R.items()}).T.fillna(0).astype(int)
kinds["rat"] = [d.split("-")[0] for d in kinds.index]
print("\nAnalysis files (obj-*.nwb, not *_behavior.nwb) by content, per rat:")
print(kinds.groupby("rat").sum().to_string())
print("\n500 Hz and 23 Hz linearizations per rat:",
      {rat: (sum(R[d]["n_lin_500"] for d in R if d.startswith(rat)),
             sum(R[d]["n_lin_23"] for d in R if d.startswith(rat))) for rat in ct.RATS})

Each run epoch has two linearizations, at the camera rate (about 23 Hz) and upsampled to 500 Hz.
The 500 Hz one is used (Methods, Change of mind time detection). Rat M's days hold each 500 Hz
linearization twice; the cell below compares the two copies.

In [ ]:
dups = pd.DataFrame([x for r in R.values() for x in r["dups"]])
print(f"run epochs with more than one 500 Hz linearization: {len(dups)}")
if len(dups):
    dups["rat"] = dups.file_a.str.slice(4, 9)
    print(dups.groupby("rat").identical.agg(["size", "sum"]).rename(columns={"size": "pairs", "sum": "identical"}).to_string())
print("\nStatescript log names (rat M's are spelled 'syayescript'):")
print({d.split("-")[0]: sorted({n.split(" ")[0] for n in R[d]["statescript_names"]}) for d in R})

In [ ]:
checks = pd.concat([r["checks"] for r in R.values()], ignore_index=True)
print("Run epochs per rat and day, and trials built from the digital events:")
print(checks.groupby(["rat", "day"]).agg(sessions=("epoch", "size"), trials=("trials", "sum"),
                                          rewarded=("rewarded", "sum"),
                                          ss_outer_rewards=("ss_outer_rewards", "sum"),
                                          ss_outerCount=("ss_outerCount", "sum")).to_string())
ss = checks[checks.ss_outerCount > 0]
print(f"\nAgainst the Statescript logs (rats E, J, K, L; {len(ss)} sessions): rewarded trials equal the "
      f"logged outer rewards in {np.mean(ss.rewarded == ss.ss_outer_rewards):.3f} of sessions "
      f"(largest difference {np.abs(ss.rewarded - ss.ss_outer_rewards).max()}); trials equal the logged home "
      f"rewards in {np.mean(ss.trials == ss.ss_home_rewards):.3f} (largest difference "
      f"{np.abs(ss.trials - ss.ss_home_rewards).max()}); the logged outerCount exceeds the trials by a median of "
      f"{np.median(ss.ss_outerCount - ss.trials):.0f}.")
print("Sessions with fewer than 20 trials:")
print(checks[checks.trials < ct.MIN_TRIALS][["rat", "day", "epoch", "trials"]].to_string(index=False))

Clock check. Each outer well's track segment is read from where the head is at that well's
pokes; at a trial's final poke the head should be on the final arm's segment, and 10 s before
the poke (on the way out from home) it should not. The contrast is the check that the digital
events and the linearized position share a clock.

In [ ]:
print("Segment of each outer well (from pokes), per rat-day:")
print(pd.DataFrame({d: r["seg_of_arm"] for d, r in R.items()}).T.drop_duplicates().to_string())
seg_frac = pd.DataFrame([{"day": d, "arm": a, "frac": v["frac"]} for d, r in R.items()
                         for e in r["segmap"].values() for a, v in e.items()])
print(f"fraction of a well's pokes made on its modal segment, per epoch: median {seg_frac.frac.median():.3f}, "
      f"min {seg_frac.frac.min():.3f}")
print("\nFraction of trials with the head on the final arm's segment, at the final poke and shifted by 10 s:")
print(checks.groupby("rat")[["end_on_final_arm", "on_final_arm_10s_before", "on_final_arm_10s_after",
                             "median_depth_at_poke"]].mean().round(3).to_string())
lens = pd.DataFrame({d: r["arm_len"] for d, r in R.items()}).T
print(f"\nOuter arm length in linearized units (cm): min {lens.min().min():.1f}, max {lens.max().max():.1f} "
      f"(Methods: 96 cm from the beginning of the arm to the end of the reward station)")
print(f"fraction of 500 Hz linearized samples that are NaN, per epoch: max {checks.lin_nan_frac.max():.4f}")

In [ ]:
T = pd.concat([r["trials"] for r in R.values()], ignore_index=True)
TP = pd.concat([r["trials_prop"] for r in R.values()], ignore_index=True)
T["dayidx"] = T.groupby("rat").day.transform(lambda s: s.rank(method="dense").astype(int) - 1)
TP["dayidx"] = T["dayidx"].values
T["nses"] = T.groupby(["rat", "day", "epoch"]).trial.transform("size")
TP["nses"] = T["nses"].values
assert (T.start.values == TP.start.values).all()
RATS = ["M", "E", "J", "K", "L"]
# sessions with at least 20 trials (claims file, opening comment, point 2)
TQ_mask = T.nses >= ct.MIN_TRIALS

# The rewarded cycle per rat, read from the reward log (claims file, opening comment).
seqs = {}
for rat, g in T.groupby("rat"):
    cyc, sup = ct.infer_sequence(g)
    seqs[rat] = cyc
    a = 1
    order = [a]
    for _ in range(4):
        a = cyc[a]
        order.append(a)
    print(f"rat {rat}: rewarded cycle {'-'.join(map(str, order))}; share of rewards followed by the cycle's "
          f"next arm as next reward: {min(sup.values()):.2f} to {max(sup.values()):.2f}")


def add_correct_arm(df):
    """Correct arm of each trial: the cycle's next arm after the last rewarded arm of the session."""
    out = np.full(len(df), np.nan)
    for _, g in df.groupby(["rat", "day", "epoch"], sort=False):
        last = None
        for i, (rat, fin, rew) in zip(g.index, zip(g.rat, g.final, g.rewarded)):
            if last is not None:
                out[i] = seqs[rat][last]
            if rew:
                last = int(fin)
    df["correct_arm"] = out
    return df


T = add_correct_arm(T)
TP["correct_arm"] = T.correct_arm.values
ok = T.correct_arm.notna()
print(f"\nTrials with a correct arm defined (a reward earlier in the session): {ok.sum()} of {len(T)}; "
      f"rewarded exactly when the final arm is the correct arm in {np.mean(T[ok].rewarded == (T[ok].final == T[ok].correct_arm)):.4f}")
TQ = T[TQ_mask].copy()
TPQ = TP[TQ_mask].copy()
print(f"sessions with fewer than {ct.MIN_TRIALS} trials: {T[~TQ_mask].groupby(['day', 'epoch']).ngroups}, holding {int((~TQ_mask).sum())} trials; left out from here on except in fig1e-trial-counts, which counts both ways")


The reward log gives 3-4-2-1-3 for rats M, E and J and 1-2-4-3-1 for rats K and L. The Results say
1-2-4-3-1 in two rats and 3-4-2-1-3 in three, which fits; the Figure 1B legend gives 3-4-2-1-3 for
rat E, which fits; Figure S1E labels Seq2 (1-2-4-3-1) for M, E and J, which does not, unless the
arm numbers in the files differ from the figure's. The files label the epochs of K and L "Rev2" and
those of M, E and J "Seq2".

## fig1c-learning

> All rats learned the spatial sequence task over a period of 5 to 19 days, the percentage of
> consecutive trials conforming to the correct sequence transitions rising over the days analysed.

In [ ]:
U = T[T.nses >= ct.MIN_TRIALS].copy()
U["next"] = U.groupby(["rat", "day", "epoch"]).final.shift(-1)
U = U.dropna(subset=["next"])
U["seqtrans"] = np.array([seqs[r][int(a)] == int(b) for r, a, b in zip(U.rat, U.final, U.next)], dtype=float)
lc = U.groupby(["rat", "dayidx"]).seqtrans.mean().unstack(0)[RATS]
print("Fraction of adjacent-trial transitions that are transitions of the rat's cycle, per day (sessions >= 20 trials):")
print(lc.round(3).to_string())
rows = []
for rat in RATS:
    v = lc[rat].dropna()
    rho, p = stats.spearmanr(v.index, v.values)
    rows.append({"rat": rat, "days": len(v), "first": v.iloc[0], "peak_day": int(v.idxmax()), "peak": v.max(),
                 "last_day_is_peak": int(v.idxmax()) == len(v) - 1, "spearman_rho": rho, "spearman_p": p,
                 "holds": (v.max() > v.iloc[0]) and (v.max() > 0.25) and rho > 0})
fig1c = pd.DataFrame(rows)
print(fig1c.round(4).to_string(index=False))
print(f"rats meeting the criterion: {fig1c.holds.sum()} of 5")
nrep = int((U.final == U.next).sum())
print(f"transitions to the same arm: {nrep} of {len(U)}; without repeats chance is 1/3, not the paper's 1/4: "
      f"first days above 1/3 in {int((fig1c['first'] > 1 / 3).sum())} rats, peak days above 1/3 in {int((fig1c.peak > 1 / 3).sum())}")
fig, ax = plt.subplots(figsize=(5, 3))
for rat in RATS:
    v = lc[rat].dropna()
    ax.plot(v.index - v.idxmax(), 100 * v.values, marker="o", label=rat)
ax.axhline(25, ls="--", c="gray")
ax.set_xlabel("days from peak performance")
ax.set_ylabel("% transitions of the cycle")
ax.legend(fontsize=7)
print(f"plotted range: {100 * lc.min().min():.1f} to {100 * lc.max().max():.1f} %")

**Result: reproduced.** In every rat the value on its peak day is above the first day's and above
25%, and the Spearman correlation with day is positive (table above). The dandiset holds 5, 9, 12,
8 and 6 days for rats M, E, J, K and L, against about 6, 9, 11, 8 and 6 in Figure 1C. The rats
almost never repeat an arm (52 of 14708 transitions), so chance without repeats is 1/3, not the
25% the criterion uses: the first day is above 1/3 in 3 rats and the peak day in all 5.

## fig1e-trial-counts

> The dataset analysed holds 2370, 3554, 4628, 3779 and 2087 trials for rats M, E, J, K and L, of
> which 107, 225, 138, 315 and 309 are COM trials, hundreds of COM trials per subject.

In [ ]:
# expected values and tolerances of the claims file
exp_trials = {"M": 2370, "E": 3554, "J": 4628, "K": 3779, "L": 2087}
tol_trials = {"M": 237, "E": 355, "J": 463, "K": 378, "L": 209}
exp_com = {"M": 107, "E": 225, "J": 138, "K": 315, "L": 309}
tol_com = {"M": 11, "E": 23, "J": 14, "K": 32, "L": 31}
readings = []
for theta, df in (("a, 10 cm", T), ("b, 0.1 of arm length", TP)):
    for sel, d in (("i, all trials", df), ("ii, sessions >= 20 trials", df[df.nses >= ct.MIN_TRIALS])):
        n = d.groupby("rat").size()
        c = d.groupby("rat").com.sum()
        within = [abs(n[r] - exp_trials[r]) <= tol_trials[r] for r in RATS] + \
                 [abs(c[r] - exp_com[r]) <= tol_com[r] for r in RATS]
        readings.append({"threshold": theta, "trials": sel,
                         **{f"n_{r}": n[r] for r in RATS}, **{f"com_{r}": int(c[r]) for r in RATS},
                         "within_tol": sum(within), "min_com": int(c.min())})
fig1e = pd.DataFrame(readings)
print("expected trials", exp_trials, "\nexpected COM trials", exp_com)
print(fig1e.to_string(index=False))
print("\nTrials per rat relative to the paper (reading a-i):",
      {r: round(fig1e.iloc[0][f"n_{r}"] / exp_trials[r], 3) for r in RATS})
print("COM trials as a share of trials, dataset (reading a-i) and paper:",
      {r: (round(fig1e.iloc[0][f"com_{r}"] / fig1e.iloc[0][f"n_{r}"], 3), round(exp_com[r] / exp_trials[r], 3)) for r in RATS})
# Where the shortfall is: trials per day, against the days of Figure 1C, and run epochs not tagged "Session"
per_day = T.groupby(["rat", "dayidx"]).size().unstack(0)[RATS]
print("\nTrials per day in the dandiset (all sessions):")
print(per_day.fillna(0).astype(int).to_string())
print("days per rat:", per_day.notna().sum().to_dict(), "; Figure 1C shows about 6, 9, 11, 8 and 6 for M, E, J, K, L")
untagged = sorted({(ct.RATS[d.split("-")[0]], tag) for d, r in R.items() for tag in r["epochs"].tag
                   if "Session" not in tag and "Sleep" not in tag})
print("epochs tagged neither Session nor Sleep:", untagged if untagged else "none")
print("run epochs per rat:", {r: int(n) for r, n in T.groupby("rat").apply(lambda g: g.groupby(["day", "epoch"]).ngroups).items()})


In [ ]:
# The COM-time rule of the round-1 revision (a reversal counts once it ends after the head has passed the
# threshold) against the earlier rule (the first reversal of the visit), 10 cm, all sessions
EV0 = pd.concat([r["events_earlier_rule"] for r in R.values() if len(r["events_earlier_rule"])], ignore_index=True)
EVc = pd.concat([r["events"] for r in R.values() if len(r["events"])], ignore_index=True)
cmp = pd.DataFrame({"com_trials": T.groupby("rat").com.sum(), "com_trials_earlier_rule": T.groupby("rat").com_earlier_rule.sum(),
                    "events": EVc.groupby("rat").size(), "events_earlier_rule": EV0.groupby("rat").size()}).loc[RATS]
mm = EVc.merge(EV0, on=["day", "epoch", "trial_start", "arm", "t_enter"], how="outer", suffixes=("", "_old"), indicator=True)
moved = mm[(mm._merge == "both") & (np.abs(mm.t_rev - mm.t_rev_old) > 1e-9)]
cmp["events_time_moved"] = moved.groupby("rat").size().reindex(RATS).fillna(0).astype(int)
print(cmp.to_string())
print(f"events found under one rule only: {int((mm._merge != 'both').sum())}; moved COM times, shift median "
      f"{(moved.t_rev - moved.t_rev_old).median() if len(moved) else float('nan'):.3f} s")


**Result: not reproduced.** The best readings (threshold b, 9.6 cm) put 4 of the 10 counts within
tolerance, the trial and COM counts of rats E and J; reading a puts 3. The criterion asks for at
least 5 for a partial and 8 for a full reproduction. Every rat has more than 100 COM trials under
every reading (minimum 120). The dandiset holds 0.88, 0.78 and 0.87 of the paper's trials for rats
M, K and L, and the COM share of trials is close to the paper's for rats E, J, K and L (0.056
against 0.063, 0.028 against 0.030, 0.076 against 0.083, 0.132 against 0.148), so most of the
shortfall is in trials missing from the dandiset; rat M has more COM trials than the paper (120
against 107). Where the trials are missing cannot be told from the files: rat M has 5 days of about
400 trials each against the paper's 6 days, which would account for its shortfall, while rats K
and L have the 8 and 6 days of Figure 1C, and no epoch is tagged other than "Session" or "Sleep".
The rule for the COM time adopted after the first verification (a reversal counts once it ends
after the head has passed the threshold) gives the same COM trials and events as the earlier rule
(the first reversal of the visit) in every rat; it moves 11 COM times (1, 1, 4, 0 and 5 in rats M,
E, J, K and L), by a median 1.0 s.

## fig1d-initial-switch-bias

> Early in learning every rat was biased to side-switching transitions, with a ratio of switch-side
> to same-side transition probabilities greater than 2 (the ratio for a random choice).

In [ ]:
V = U[U.final != U.next].copy()
V["switch"] = [ct.is_switch(a, b) for a, b in zip(V.final, V.next)]
cnt = V.groupby(["rat", "dayidx"]).switch.agg(["sum", "size"])
cnt["ratio"] = cnt["sum"] / (cnt["size"] - cnt["sum"])
ratio = cnt.ratio.unstack(0)[RATS]
print("P(switch-side)/P(same-side) per day (adjacent trials to different arms, sessions >= 20 trials):")
print(ratio.round(2).to_string())
first = ratio.iloc[0]
first2 = ratio.iloc[:2].mean()
for name, v in (("a, first day", first), ("b, mean of first two days", first2)):
    k = int((v > 2).sum())
    print(f"reading {name}: {v.round(2).to_dict()}; above 2 in {k} of 5; one-sided binomial p = "
          f"{stats.binomtest(k, 5, 0.5, alternative='greater').pvalue:.3f}")
print("Transitions excluded because both trials went to the same arm:", int((U.final == U.next).sum()), "of", len(U))

**Result: reproduced.** On the first analysed day the ratio is 5.53, 4.23, 3.76, 2.31 and 2.25
for rats M, E, J, K and L, above 2 in all five (one-sided binomial p = 0.031); the mean of the
first two days is also above 2 in all five.

## fig1d-switch-bias-declines

> As learning progressed the switch-side to same-side ratio fell to about 1.5, with each day in the
> maze multiplying the bias by an estimated 0.92.

In [ ]:
X = pd.get_dummies(V.rat).astype(float)[RATS]
X["day"] = V.dayidx.astype(float)
m = sm.Logit(V.switch.astype(float), X).fit(disp=0)
ci = np.exp(m.conf_int().loc["day"]).values
print(f"(a/b, i) logistic regression on day index with one indicator per rat, no intercept: n = {len(V)} transitions; "
      f"day effect exp(b) = {np.exp(m.params['day']):.4f} (95% CI {ci[0]:.4f} to {ci[1]:.4f}), p = {m.pvalues['day']:.2e}")
X2 = X.copy()
X2["day"] = (V.dayidx - V.groupby("rat").dayidx.transform("max")).astype(float)
m2 = sm.Logit(V.switch.astype(float), X2).fit(disp=0)
print(f"(a) days from the last day: exp(b) = {np.exp(m2.params['day']):.4f}, p = {m2.pvalues['day']:.2e}")
Xb = sm.add_constant(X.drop(columns="M"))
mb = sm.Logit(V.switch.astype(float), Xb).fit(disp=0)
print(f"intercept and four rat indicators: exp(b) = {np.exp(mb.params['day']):.4f}, p = {mb.pvalues['day']:.2e}")
for rat in RATS:
    v = V[V.rat == rat]
    mr = sm.Logit(v.switch.astype(float), sm.add_constant(v.dayidx.astype(float))).fit(disp=0)
    print(f"  rat {rat} alone: exp(b) = {np.exp(mr.params['dayidx']):.3f}, p = {mr.pvalues['dayidx']:.2e}")
peak = {r: int(lc[r].dropna().idxmax()) for r in RATS}
pr = {r: ratio.loc[peak[r], r] for r in RATS}
pooled_peak = V[[d == peak[r] for r, d in zip(V.rat, V.dayidx)]]
pooled_ratio = pooled_peak.switch.sum() / (~pooled_peak.switch).sum()
print(f"\nratio on each rat's peak day: {dict((r, round(v, 2)) for r, v in pr.items())}")
print(f"final ratio (a) mean over rats = {np.mean(list(pr.values())):.3f}; (b) pooled over rats = {pooled_ratio:.3f}")
last = {r: ratio[r].dropna().iloc[-1] for r in RATS}
print(f"ratio on each rat's last day: {dict((r, round(v, 2)) for r, v in last.items())}, mean {np.mean(list(last.values())):.3f}")

**Result: partially reproduced.** Each day multiplies the odds of a switch-side transition by
0.933 (95% CI 0.920 to 0.946, p = 2.3e-23), within 0.92 plus or minus 0.092, under every coding of
the day index and the rat indicators. The ratio on the peak day is 1.85 as a mean over rats and
1.84 pooled, outside 1.5 plus or minus 0.15 under both readings. The peak day is each rat's last
day in the dandiset. Rat K alone shows no significant decline (0.984, p = 0.38).

## figs1f-multiple-com

> In most COM trials the rats changed their minds once (entering two choice arms), but trials with
> several COM events (more than two arms entered) occurred in every rat.

In [ ]:
for name, df in (("a, 10 cm", TQ), ("b, 0.1 of arm length (9.6 cm)", TPQ)):
    c = df[df.com]
    tab = c.groupby("rat").agg(com_trials=("com", "size"), one_event=("n_com", lambda s: (s == 1).mean()),
                               two_plus_events=("n_com", lambda s: int((s >= 2).sum())),
                               two_plus_arms=("n_com_arm", lambda s: int((s >= 2).sum())),
                               max_events=("n_com", "max")).loc[RATS]
    tab["holds"] = (tab.one_event > 0.5) & (tab.two_plus_events >= 1)
    print(f"reading {name}:")
    print(tab.round(3).to_string())

**Result: reproduced.** In every rat more than half of COM trials have one COM event, and every rat
has trials with two or more, under both thresholds (tables above); trials with more than two arms
entered (two or more COM arms) also occur in every rat.

## fig1g-com-corrective

> COM trials were corrective; the final choice of a COM trial was rewarded more often than the
> initial choice would have been, in each rat.

In [ ]:
def corrective(df, label):
    c = df[df.com & df.correct_arm.notna()].copy()
    c["init_correct"] = (c.initial == c.correct_arm).astype(float)
    c["final_correct"] = c.rewarded.astype(float)
    daily = c.groupby(["rat", "dayidx"])[["init_correct", "final_correct"]].mean()
    rows = []
    for rat in RATS:
        d = daily.loc[rat]
        p = stats.ranksums(d.final_correct, d.init_correct).pvalue
        rows.append({"rat": rat, "days": len(d), "com_trials": int((c.rat == rat).sum()),
                     "initial_median": d.init_correct.median(), "final_median": d.final_correct.median(),
                     "final_higher": d.final_correct.median() > d.init_correct.median(), "ranksum_p": p})
    out = pd.DataFrame(rows)
    out["holds"] = out.final_higher & (out.ranksum_p < 0.05)
    print(f"reading {label} (COM trials with a correct arm defined; dropped as first in session before a reward: "
          f"{int((df.com & df.correct_arm.isna()).sum())}):")
    print(out.round(4).to_string(index=False))
    return out, daily


g1, daily_g = corrective(TQ, "a, 10 cm")
g1b, _ = corrective(TPQ, "b, 0.1 of arm length (9.6 cm)")
print(f"rats meeting the criterion: {g1.holds.sum()} (10 cm), {g1b.holds.sum()} (0.1 of arm)")

**Result: reproduced.** In all five rats the daily probability that the final choice was rewarded
is higher than that of the initial choice (medians 0.47 to 0.72 against 0.00 to 0.18), with
rank-sum p from 0.028 (rat M) down to below 1e-4 (rat J), the same under both thresholds.

## fig1g-com-matches-run-through

> After correction, the reward rate on COM trials did not differ from that of temporally nearby
> run-through trials.

In [ ]:
def run_through(df, label, seed):
    rng = np.random.default_rng(seed)
    sessions = [(k, g.reset_index()) for k, g in df.groupby(["rat", "day", "epoch"], sort=True)]
    pvals = {r: [] for r in RATS}
    diffs = {r: [] for r in RATS}
    no_ctrl = 0
    for draw in range(100):
        rows = []
        for (rat, day, ep), g in sessions:
            if not g.com.any():
                continue
            ctrl = ct.nearby_control(g, rng)
            comrows = g[g.com]
            for (_, cr), j in zip(comrows.iterrows(), ctrl):
                rows.append({"rat": rat, "dayidx": cr.dayidx, "com_rew": float(cr.rewarded),
                             "ctrl_rew": float(g.rewarded.iloc[j]) if j >= 0 else np.nan})
        d = pd.DataFrame(rows)
        if draw == 0:
            no_ctrl = int(d.ctrl_rew.isna().sum())
        com_daily = d.groupby(["rat", "dayidx"]).com_rew.mean()
        ctrl_daily = d.dropna().groupby(["rat", "dayidx"]).ctrl_rew.mean()
        for rat in RATS:
            a, b = com_daily.loc[rat], ctrl_daily.loc[rat]
            pvals[rat].append(stats.ranksums(a, b).pvalue)
            diffs[rat].append(a.median() - b.median())
    out = pd.DataFrame({"rat": RATS, "median_p": [np.median(pvals[r]) for r in RATS],
                        "share_p_below_0.05": [np.mean(np.array(pvals[r]) < 0.05) for r in RATS],
                        "median_diff_COM_minus_nearby": [np.median(diffs[r]) for r in RATS]})
    print(f"reading {label}: COM trials without an eligible nearby trial (draw 0): {no_ctrl}")
    print(out.round(4).to_string(index=False))
    print("rats with a significant difference (median p < 0.05):", int((out.median_p < 0.05).sum()))
    return out


rt = run_through(TQ, "a, 10 cm", 1836)
rtb = run_through(TPQ, "b, 0.1 of arm length (9.6 cm)", 1837)


**Result: reproduced.** No rat shows a significant difference between the daily reward rate of
COM trials and that of nearby non-COM trials: the median p over 100 draws is 0.17 to 0.75 at 10 cm
(0.16 to 0.75 at 9.6 cm), and at most 22% of draws (rat K) give p < 0.05. The median differences
are -0.08 to 0.12. Each rat has only 5 to 12 daily values, so the test has little power.

## fig1h-final-choice-less-switching

> The initial choice on COM trials was heavily biased to side-switching, while the final choice
> was much less so, with a switch-side to same-side ratio close to 1.

In [ ]:
def sign_p(x, ref):
    x = np.asarray(x)
    above, below = int((x > ref).sum()), int((x < ref).sum())
    return stats.binomtest(above, above + below, 0.5).pvalue if above + below else np.nan, above, below


def fig1h_table(df, label, smooth=False):
    d = df.copy()
    d["prev"] = d.groupby(["rat", "day", "epoch"]).final.shift(1)
    c = d[d.com & d.prev.notna()].copy()
    rows = []
    for kind in ("initial", "final"):
        x = c[c[kind] != c.prev]
        x = x.assign(sw=[ct.is_switch(a, b) for a, b in zip(x.prev, x[kind])])
        cnt = x.groupby(["rat", "dayidx"]).sw.agg(["sum", "size"])
        if smooth:   # add one to both counts, so that no day is dropped
            cnt["ratio"] = (cnt["sum"] + 1) / (cnt["size"] - cnt["sum"] + 1)
        else:
            cnt["ratio"] = cnt["sum"] / (cnt["size"] - cnt["sum"])
        rows.append(cnt.ratio.rename(kind))
    daily = pd.concat(rows, axis=1).replace(np.inf, np.nan)
    out = []
    for rat in RATS:
        dd = daily.loc[rat]
        both = dd.dropna()
        t = stats.ttest_rel(both.initial, both.final) if len(both) > 1 else None
        pi2, ia, ib = sign_p(dd.initial.dropna(), 2)
        pf2, fa, fb = sign_p(dd.final.dropna(), 2)
        pf1, f1a, f1b = sign_p(dd.final.dropna(), 1)
        out.append({"rat": rat, "days_both": len(both), "dropped_days_init": int(dd.initial.isna().sum()),
                    "dropped_days_final": int(dd.final.isna().sum()),
                    "init_median": dd.initial.median(), "final_median": dd.final.median(),
                    "paired_t_p": t.pvalue if t else np.nan, "init_gt_final": both.initial.mean() > both.final.mean(),
                    "sign_init_vs2_p": pi2, "init_above2": f"{ia}/{ia + ib}",
                    "sign_final_vs2_p": pf2, "final_below2": f"{fb}/{fa + fb}",
                    "sign_final_vs1_p": pf1, "final_above1": f"{f1a}/{f1a + f1b}"})
    out = pd.DataFrame(out)
    print(f"reading {label}:")
    print(out.round(4).to_string(index=False))
    t_ok = int(((out.paired_t_p < 0.01) & out.init_gt_final).sum())
    i_ok = int(((out.sign_init_vs2_p < 0.05) & (out.init_median > 2)).sum())
    f_ok = int(((out.sign_final_vs2_p < 0.05) & (out.final_median < 2)).sum())
    print(f"paired t p < 0.01 with initial > final: {t_ok} rats; initial above 2 by sign test: {i_ok}; "
          f"final below 2 by sign test: {f_ok}")
    return out, daily


h1, daily_h = fig1h_table(TQ, "a, 10 cm")
h1b, _ = fig1h_table(TPQ, "b, 0.1 of arm length (9.6 cm)")
print("\nVariant: counts plus one, no day dropped")
h1s, _ = fig1h_table(TQ, "a, 10 cm, counts plus one", smooth=True)
h1bs, _ = fig1h_table(TPQ, "b, 9.6 cm, counts plus one", smooth=True)

**Result: not reproduced.** The initial ratio exceeds the final in all five rats, but the paired
t-test reaches p < 0.01 only in rats J and K under both thresholds (p = 0.0023 and 0.0045 at 10
cm); rats E and L have p = 0.013 and 0.014, and the criterion asks for four rats at p < 0.01, or
three for a partial reproduction. The sign tests at 10 cm hold as the legend reports: the initial
ratio is above 2 and the final below 2 with p < 0.05 in rats E, J, K and L, not in rat M. The
criterion drops days whose ratio is undefined (1 day of E and 2 of J). In a variant that adds one
to both counts, so that no day is dropped, rats J, K and L reach p < 0.01 (E has p = 0.010), which
would be a partial reproduction; the criterion's rule decides the outcome.

## fig1h-final-ratio-near-one

> The final choice of COM trials did not differ from a switch-side to same-side ratio of 1 in any
> rat.

In [ ]:
print(h1[["rat", "final_median", "final_above1", "sign_final_vs1_p"]].round(4).to_string(index=False))
print("rats differing from 1 (sign test p < 0.05), 10 cm:", int((h1.sign_final_vs1_p < 0.05).sum()),
      "; 0.1 of arm:", int((h1b.sign_final_vs1_p < 0.05).sum()))

**Result: reproduced.** No rat's daily final-choice ratio differs from 1 by the sign test (p =
0.18 to 1.0); the medians are 0.93 to 1.41. For rat M (5 days) the test cannot reach p < 0.05
(smallest two-sided p 0.0625), so its part of the result is weak.

## The triggered decode tables (Figure 3)

The paper's code (Gu2026, commit 0f0c9a7, `changeOfMind_triggered_position.py` and
`changeOfMind_triggered.py`) writes these tables: per row, a window of the linearized head position
and of the decoded position, with `triggered_trial_info` = (trial, arm). Which parameters made each
file (trigger, window, decoder, MAP or mean) is kept in the lab's database and not in the file. The
cells below identify each file by aligning its trigger times with this entry's COM times (both
thresholds) and outer pokes, by its window, and by whether its positions are relative to the
trigger (a table restricted to the COM arm, where the code sets the decode to NaN when 0.2 or less
of the posterior is in the arm). The arm regions on the linearization are the code's
(`loc1d_to_baseoff`), checked against the deposited linearization. Two further cells look at the
tables that match neither trigger ("other"), and at which decoder made which table, from how often
two tables that share samples have the same decode.


In [ ]:
print("Arm base on the linearization: the code's region against the smallest head position on each arm's segment, "
      "median over days:")
starts = pd.DataFrame({d: r["arm_start"] for d, r in R.items()}).T
starts["rat"] = [ct.RATS[d.split("-")[0]] for d in starts.index]
print(pd.concat([starts.groupby("rat").median().loc[RATS].round(1),
                 pd.DataFrame([ct.ARM_REGIONS[1:, 0].round(1)], index=["code"], columns=[1, 2, 3, 4])]).to_string())
EV = pd.concat([r["events"] for r in R.values() if len(r["events"])], ignore_index=True)
EVP = pd.concat([r["events_prop"] for r in R.values() if len(r["events_prop"])], ignore_index=True)
jobs3 = []
for k, d in enumerate(sorted(R)):
    com_t = np.concatenate([EV[EV.day == d].t_rev.values, EVP[EVP.day == d].t_rev.values])
    jobs3.append((assets[assets.day == d], R[d]["epochs"], com_t, T[T.day == d].end.values, 1836 + k))
t0 = time.time()
F3 = dict(zip(sorted(R), ct.processes(ct.fig3_day, jobs3, workers=16)))
print(f"\ntriggered tables read for {len(F3)} days in {time.time() - t0:.0f} s")
FILES = pd.concat([o["files"].assign(day=d, rat=ct.RATS[d.split("-")[0]]) for d, o in F3.items() if len(o["files"])],
                  ignore_index=True)
print(f"unreadable tables: {int((FILES.trigger == 'unreadable').sum())}")
print(f"{len(FILES)} triggered tables. By trigger (rows mostly within {ct.TRIG_TOL_S} s of a COM time of this entry, "
      "of an outer poke, or neither), window and restriction to the arm:")
summ = FILES.groupby(["trigger", "window", "segment"]).agg(
    files=("file", "size"), rows=("rows", "sum"), frac_rows_aligned_com=("frac_com", "median"),
    frac_rows_aligned_poke=("frac_poke", "median"), median_d_com_s=("median_d_com", "median"),
    median_d_poke_s=("median_d_poke", "median"), rel_start_s=("rel_min", "median"), rel_end_s=("rel_max", "median"),
    nan_decode=("nan_decode", "median"))
print(summ.round(4).to_string())
print("\nfiles per rat and kind:")
print(pd.crosstab(FILES.rat, FILES.trigger + "/" + FILES.window + "/" + np.where(FILES.segment, "arm", "track")).loc[RATS].to_string())


In [ ]:
# The tables that match neither trigger ("other"): where their triggers fall
OR = pd.DataFrame([dict(m, day=d, rat=ct.RATS[d.split("-")[0]]) for d, o in F3.items() for m in o["other_rows"]])
EV0 = pd.concat([r["events_earlier_rule"] for r in R.values() if len(r["events_earlier_rule"])], ignore_index=True)
OR["d_com_earlier_rule"] = [float(ct._nearest(EV0[EV0.day == d].t_rev.values, np.array([t]))[0]) for d, t in zip(OR.day, OR.t_trig)]
vis = {d: g[["t_enter", "t_exit"]].values for d, g in EV.groupby("day")}
OR["in_com_visit"] = [bool(d in vis and np.any((vis[d][:, 0] <= t) & (vis[d][:, 1] >= t))) for d, t in zip(OR.day, OR.t_trig)]
OF = FILES[FILES.trigger == "other"]
print(f"tables matching neither trigger: {len(OF)}, {int(OF.rows.sum())} rows; files per rat "
      f"{OF.groupby('rat').size().reindex(RATS).fillna(0).astype(int).to_dict()}; rows per file median {OF.rows.median():.0f} "
      f"[{OF.rows.min()}, {OF.rows.max()}]; share of each file's rows at a COM time (within 20 ms): median {OF.frac_com.median():.2f}")
print("their rows, per rat: share with the trigger within 20 ms of a COM time (this entry's rule or the earlier one), "
      "within 1 s and 5 s of one, inside the arm visit of one of this entry's COM events (10 cm), and within 20 ms of an outer poke")
print(OR.groupby("rat").agg(rows=("d_com", "size"), com_20ms=("d_com", lambda s: (s < ct.TRIG_TOL_S).mean()),
                            earlier_rule_20ms=("d_com_earlier_rule", lambda s: (s < ct.TRIG_TOL_S).mean()),
                            com_1s=("d_com", lambda s: (s < 1).mean()), com_5s=("d_com", lambda s: (s < 5).mean()),
                            in_com_visit=("in_com_visit", "mean"), poke_20ms=("d_poke", lambda s: (s < ct.TRIG_TOL_S).mean()),
                            median_d_com_s=("d_com", "median")).reindex(RATS).round(3).to_string())
print(f"rows over 60 s from any COM time and poke: {int(((OR.d_com > 60) & (OR.d_poke > 60)).sum())}")


In [ ]:
# Which decoder made which table. At the samples two tables of a day share (same trigger, same times),
# how often their decodes agree to within 1 cm.
P = pd.concat([o["pairs"].assign(day=d) for d, o in F3.items() if len(o["pairs"])], ignore_index=True)
kind = dict(zip(FILES.file, np.where(FILES.segment, "arm", "track")))
P["ka"], P["kb"] = P.file_a.map(kind), P.file_b.map(kind)
aa = P[(P.ka == "arm") & (P.kb == "arm") & (P.samples >= 100)]
print(f"pairs of arm-restricted tables sharing samples: {len(aa)}; share of samples that agree: median {aa.frac_equal.median():.4f}, "
      f"min {aa.frac_equal.min():.4f}; pairs below 0.99: {int((aa.frac_equal < 0.99).sum())}")
tr = FILES[~FILES.segment & (FILES.window == "both")]
print(f"whole-track tables around a trigger: {len(tr)}; with an arm-restricted table sharing 100 or more samples on the arm: "
      f"{int(tr.arm_table_agreement.notna().sum())}")
print("their agreement with the arm-restricted tables, on the arm (decoder A at 0.99 or more):")
print(pd.cut(tr.arm_table_agreement, [0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0001]).value_counts().sort_index().to_string())
tt = P[(P.ka == "track") & (P.kb == "track") & (P.samples >= 100)].copy()
tt["agree_a"], tt["agree_b"] = tt.file_a.map(FILES.set_index("file").arm_table_agreement), tt.file_b.map(FILES.set_index("file").arm_table_agreement)
both_known = tt.dropna(subset=["agree_a", "agree_b"])
one_a = ((both_known.agree_a >= 0.99) != (both_known.agree_b >= 0.99)).sum()
print(f"pairs of whole-track tables with the same triggers: {len(tt)}; their decodes agree on a median {tt.frac_equal.median():.3f} of samples "
      f"({tt.frac_equal_on_arm.median():.3f} where both are on the arm); pairs where both members can be compared with an "
      f"arm-restricted table: {len(both_known)}, with exactly one member at 0.99 or more: {int(one_a)}")


## fig3b-extended-before-stopping

> Before stopping, COM trials showed more extended hippocampal representations within the arm than
> outbound run-through trials over matched arm regions.

COM rows: the COM-triggered tables restricted to the arm, from arm entry to stopping. Control rows:
the tables triggered at outer pokes, 2 s before the poke (outbound runs), restricted to the arm.
Per row the largest |head - decode| at 4 cm/s or more; each control row over a head range drawn at
random from the same day's COM rows (figure4c.make_distance_plot and return_distance_day).


In [ ]:
CP = pd.DataFrame([dict(m, day=d, rat=ct.RATS[d.split("-")[0]]) for d, o in F3.items() for m in o["com_pre"]])
PP = pd.DataFrame([dict(m, day=d, rat=ct.RATS[d.split("-")[0]]) for d, o in F3.items() for m in o["poke_pre"]])
print(f"COM rows (COM-triggered, arm-restricted, before stopping): {len(CP)}; with a distance (moments at >= 4 cm/s "
      f"and >0.2 posterior in the arm): {CP.d_all.notna().sum()}")
print(f"control rows (outer-poke-triggered, 2 s before the poke, arm-restricted): {len(PP)}; with a distance over the "
      f"drawn COM range: {PP.d_matched.notna().sum()}; over the whole window: {PP.d_all.notna().sum()}")
rows = []
for rat in RATS:
    a = CP[CP.rat == rat].d_all.dropna()
    b = PP[PP.rat == rat].d_matched.dropna()
    b2 = PP[PP.rat == rat].d_all.dropna()
    rows.append({"rat": rat, "n_com": len(a), "n_ctrl_matched": len(b), "com_median_cm": a.median(),
                 "ctrl_matched_median_cm": b.median(),
                 "p_greater_matched": stats.ranksums(a, b, alternative="greater").pvalue if len(a) and len(b) else np.nan,
                 "ctrl_all_median_cm": b2.median(),
                 "p_greater_unmatched": stats.ranksums(a, b2, alternative="greater").pvalue if len(a) and len(b2) else np.nan})
f3b = pd.DataFrame(rows)
f3b["holds"] = (f3b.com_median_cm > f3b.ctrl_matched_median_cm) & (f3b.p_greater_matched < 0.01)
print(f3b.round(4).to_string(index=False))
print(f"rats with COM greater at p < 0.01 (matched ranges, the paper's procedure): {int(f3b.holds.sum())} of 5")


In [ ]:
# The same test over 20 further draws of the control arm ranges, each with its own seed
rows = []
for k in range(1, ct.N_DRAWS + 1):
    for rat in RATS:
        a = CP[CP.rat == rat].d_all.dropna()
        b = PP[PP.rat == rat][f"d_matched_{k}"].dropna()
        rows.append({"draw": k, "rat": rat, "holds": bool(a.median() > b.median() and stats.ranksums(a, b, alternative="greater").pvalue < 0.01),
                     "p": stats.ranksums(a, b, alternative="greater").pvalue})
DX = pd.DataFrame(rows)
print("per rat over the 20 draws: draws with COM greater at p < 0.01, and the range of p")
print(DX.groupby("rat").agg(draws_holding=("holds", "sum"), p_min=("p", "min"), p_max=("p", "max")).loc[RATS].to_string(float_format=lambda v: f"{v:.3g}"))
print("rats holding per draw (count of draws):", DX.groupby("draw").holds.sum().value_counts().sort_index().to_dict())
pm = DX.groupby("rat").p.median().loc[RATS]
print("median p over the 20 draws, per rat:", pm.map(lambda v: f"{v:.2g}").to_dict(), f"; rats below 0.01: {int((pm < 0.01).sum())}")
# with the rows of the arm-restricted tables before stopping that match neither trigger added to the COM rows
OP = pd.DataFrame([dict(m, day=d, rat=ct.RATS[d.split("-")[0]]) for d, o in F3.items() for m in o["other_pre"]])
CPo = pd.concat([CP, OP], ignore_index=True)
print(f"\nrows of the other tables before stopping: {len(OP)}, per rat {OP.groupby('rat').size().reindex(RATS).fillna(0).astype(int).to_dict()}")
rows = []
for k in range(0, ct.N_DRAWS + 1):
    col = "d_matched" if k == 0 else f"d_matched_{k}"
    for rat in RATS:
        a = CPo[CPo.rat == rat].d_all.dropna()
        b = PP[PP.rat == rat][col].dropna()
        p_ = stats.ranksums(a, b, alternative="greater").pvalue
        rows.append({"draw": k, "rat": rat, "p": p_, "holds": bool(a.median() > b.median() and p_ < 0.01)})
DXo = pd.DataFrame(rows)
print("first draw, p per rat:", DXo[DXo.draw == 0].set_index("rat").p.map(lambda v: f"{v:.2g}").to_dict(),
      f"; rats holding {int(DXo[DXo.draw == 0].holds.sum())}")
print("further draws, rats holding per draw (count of draws):", DXo[DXo.draw > 0].groupby("draw").holds.sum().value_counts().sort_index().to_dict(),
      "; median p per rat:", DXo[DXo.draw > 0].groupby("rat").p.median().loc[RATS].map(lambda v: f"{v:.2g}").to_dict())


**Result: partially reproduced.** The COM rows are the paper's own: the tables triggered at COM time,
restricted to the arm, from arm entry to stopping (908 rows; in a median 90% of rows per file the
trigger lies within 20 ms of one of this entry's COM times, at a median distance of 2 ms). Over
matched arm ranges the largest decode-to-head distance is greater on COM trials than on outbound
run-throughs with p < 0.01 in rats M, J, K and L, and not in E (p = 0.053, medians 12.7 against
12.5 cm). Only the criterion's reading (b, moments with more than 0.2 of the posterior in the arm)
and window (i, entry to stopping) can be run: the arm-restricted tables set the other moments to
NaN, and no table holds the 5 s before stopping for the controls. Without matching the control's
arm range the control distances are larger (p = 1.0 in all five rats), so the result depends on
the matching, as the code does it. The arm range of each control row is one random draw, as in the
code, and the outcome depends on it: over 20 further draws all five rats hold in 11, four in 8 and
three in 1 (rats E and J each fail in 5), and the median p over the draws is below 0.01 in every
rat (E 0.007, J 0.0045). Adding the 48 rows of the arm-restricted tables before stopping that match
neither trigger (14 of rat E, 23 of rat J) to the COM rows, three rats hold in the first draw (E p
= 0.088, J 0.013), and over the further draws five rats hold in 3, four in 11 and three in 6, with
a median p of 0.014 for E. With all of the paper's COM tables the result is partial in most draws,
so the outcome is kept.


## fig3b-distance-predicts-com

> Longer decoded distances before stopping predicted that a trial would have an upcoming COM event,
> with an odds fold change of 1.06 per unit distance.


In [ ]:
G = pd.concat([CP.assign(com=1.0, dist=CP.d_all), PP.assign(com=0.0, dist=PP.d_matched)], ignore_index=True)
for name, cols in (("a, distance alone", ["dist"]), ("b, distance and speed before stopping", ["dist", "speed"])):
    g = G.dropna(subset=cols)
    X = pd.concat([pd.get_dummies(g.rat).astype(float)[RATS], g[cols].astype(float)], axis=1)
    m = sm.Logit(g.com, X).fit(disp=0)
    ci = np.exp(m.conf_int().loc["dist"]).values
    print(f"reading {name}: n = {len(g)} ({int(g.com.sum())} COM); odds fold change per cm "
          f"{np.exp(m.params['dist']):.4f} (95% CI {ci[0]:.4f} to {ci[1]:.4f}), p = {m.pvalues['dist']:.2e}")


In [ ]:
# The control arm ranges are one random draw per row; the same fits over 20 further draws, each with its own seed
rows = []
for k in range(1, ct.N_DRAWS + 1):
    Gk = pd.concat([CP.assign(com=1.0, dist=CP.d_all), PP.assign(com=0.0, dist=PP[f"d_matched_{k}"])], ignore_index=True)
    for name, cols in (("a", ["dist"]), ("b", ["dist", "speed"])):
        g = Gk.dropna(subset=cols)
        X = pd.concat([pd.get_dummies(g.rat).astype(float)[RATS], g[cols].astype(float)], axis=1)
        m = sm.Logit(g.com, X).fit(disp=0)
        rows.append({"draw": k, "reading": name, "n": len(g), "fold_change": np.exp(m.params["dist"]), "p": m.pvalues["dist"]})
DB = pd.DataFrame(rows)
for name, g in DB.groupby("reading"):
    print(f"reading {name}: fold change {g.fold_change.min():.4f} to {g.fold_change.max():.4f} (median {g.fold_change.median():.4f}); "
          f"p {g.p.min():.2e} to {g.p.max():.2e} (median {g.p.median():.2e}); draws with p < 1e-20: {int((g.p < 1e-20).sum())} of {len(g)}; "
          f"within 1.06 +- 0.02: {int(((g.fold_change - 1.06).abs() <= 0.02).sum())}; both (the full bar): "
          f"{int((((g.fold_change - 1.06).abs() <= 0.02) & (g.p < 1e-20)).sum())}")


**Result: partially reproduced.** The odds fold change of a COM per cm of decoded distance is 1.047
(95% CI 1.035 to 1.060, p = 8.5e-14) with distance alone and 1.079 (1.062 to 1.096, p = 2.6e-20)
with the speed before stopping as a second covariate. Both are within 1.06 plus or minus 0.02,
but neither reaches p < 1e-20, so the partial bar (above 1 with p < 0.05) is what holds. Over 20
further draws of the control arm ranges, reading (a) gives 1.042 to 1.056, always within tolerance,
with p from 3e-16 to 4e-12; reading (b) gives 1.069 to 1.095, with p below 1e-20 in 13 draws but
within tolerance in only 6, and no draw meets both. The partial outcome does not depend on the draw.


## fig3c-distant-after-stopping

> Representations far from the animal were much more prevalent after stopping than before; before
> stopping they rarely reached the end of the current arm or other arms.

The tables over the whole track, 5 s either side of COM time, come in pairs with the same triggers
and different decodes (the code runs two decoders, `run_time` and `all_maze`; the file does not say
which is which). Both bounds are shown, per event the file with the smaller or the larger
post-minus-pre difference, and each decoder: A, the one whose decodes equal those of the
arm-restricted tables on the arm (identification cell above), and B, the other.


In [ ]:
CB = pd.DataFrame([dict(m, day=d, rat=ct.RATS[d.split("-")[0]], file=f) for d, o in F3.items()
                   for f, v in o["com_both"].items() for m in v])
CB["event"] = CB.day + "/" + CB.trial.astype(str) + "/" + CB.t_trig.round(2).astype(str)
nfile = CB.groupby("event").file.nunique()
print(f"COM events in the whole-track tables (5 s either side of COM time): {CB.event.nunique()}; held in two files "
      f"(two decodes): {int((nfile == 2).sum())}, in one: {int((nfile == 1).sum())}")
# decoder A: the one that agrees with the arm-restricted tables (identification cell above)
agree = FILES.set_index("file").arm_table_agreement
CB["decoder"] = np.where(CB.file.map(agree) >= 0.99, "A", np.where(CB.file.map(agree).notna(), "B", "unknown"))
print("COM events by the decoders of their files:", CB.groupby("event").decoder.agg(lambda s: "+".join(sorted(s))).value_counts().to_dict())
CB["diff"] = CB.d_post - CB.d_pre
CB = CB.dropna(subset=["d_pre", "d_post"])


def bounds(cb):
    return {"lo, the file with the smaller post-minus-pre difference": cb.loc[cb.groupby("event")["diff"].idxmin()],
            "hi, the file with the larger": cb.loc[cb.groupby("event")["diff"].idxmax()],
            "A, the decoder of the arm-restricted tables": cb[cb.decoder == "A"].drop_duplicates("event"),
            "B, the other decoder": cb[cb.decoder == "B"].drop_duplicates("event")}


def f3c_table(d):
    rows = []
    for rat in RATS:
        x = d[d.rat == rat]
        t = stats.ttest_rel(x.d_post, x.d_pre) if len(x) > 1 else None
        rows.append({"rat": rat, "events": len(x), "pre_mean_cm": x.d_pre.mean(), "post_mean_cm": x.d_post.mean(),
                     "paired_t_p": t.pvalue if t else np.nan})
    out = pd.DataFrame(rows)
    out["holds"] = (out.post_mean_cm > out.pre_mean_cm) & (out.paired_t_p < 0.001)
    return out


pick = bounds(CB)
f3c = {}
for name, d in pick.items():
    out = f3c_table(d)
    f3c[name] = out
    print(f"\nreading {name}:")
    print(out.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    print(f"rats with post > pre at p < 0.001: {int(out.holds.sum())} of 5")
print(f"\nrange of the distances: pre {CB.d_pre.min():.1f} to {CB.d_pre.max():.1f} cm, post {CB.d_post.min():.1f} to {CB.d_post.max():.1f} cm")


In [ ]:
# The whole-track tables around a trigger that are not mostly at this entry's COM times ("other"):
# (1) adding their rows whose trigger is within 20 ms of a COM time of this entry, (2) adding all their rows
CO = pd.DataFrame([dict(m, day=d, rat=ct.RATS[d.split("-")[0]], file=f) for d, o in F3.items()
                   for f, v in o["com_both_other"].items() for m in v])
CO["event"] = CO.day + "/" + CO.trial.astype(str) + "/" + CO.t_trig.round(2).astype(str)
CO["decoder"] = np.where(CO.file.map(agree) >= 0.99, "A", np.where(CO.file.map(agree).notna(), "B", "unknown"))
CO["diff"] = CO.d_post - CO.d_pre
CO = CO.dropna(subset=["d_pre", "d_post"])
CO = CO[~CO.event.isin(CB.event)]
print(f"rows of the other whole-track tables with both parts, not already among the COM events: {len(CO)}, "
      f"{CO.event.nunique()} events, of which {CO[CO.aligned].event.nunique()} within 20 ms of a COM time of this entry; "
      f"per rat {CO.drop_duplicates('event').groupby('rat').size().reindex(RATS).fillna(0).astype(int).to_dict()}")
pick_other = {}
for label, extra in (("aligned rows added", CO[CO.aligned]), ("all rows added", CO)):
    for name, d in bounds(pd.concat([CB, extra], ignore_index=True)).items():
        if name[0] in "lA":
            key = f"{label}, {'decoder A' if name[0] == 'A' else 'lo'}"
            pick_other[key] = d
            out = f3c_table(d)
            print(f"\nreading {key}:")
            print(out.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
            print(f"rats with post > pre at p < 0.001: {int(out.holds.sum())} of 5")


**Result: partially reproduced.** The whole-track tables around COM time come in pairs with
different decodes. Under both bounds and under each decoder, the largest maze distance after
stopping exceeds the one before with p < 0.001 in rats M, E, K and L (paired t-test; p from 2e-11
to 8e-32 under the smaller bound, from 1e-13 to 2e-39 under decoder A), and not in rat J (26
events; p = 0.10, 0.003, 0.058 and 0.012 under the smaller and larger bounds and decoders A and B).
Rat J has few events here because 34 of its 72 whole-track tables around a trigger match neither
trigger. Adding their rows does not change this: with those within 20 ms of a COM time of this
entry, rat J has 47 events and p = 0.23 (smaller bound) and 0.048 (decoder A); with all their rows,
52 events and p = 0.19 and 0.037. The other four rats hold in every variant.


## fig3d-distant-vs-inbound

> After stopping on COM trials, representations reached more distant locations than when animals were
> at the same locations without a COM, returning from the outer well to the home arm.

Inbound rows: the tables triggered at outer pokes, 5 s after the poke, restricted to the arm, over
a head range drawn from the same day's COM post-stopping ranges.


In [ ]:
PO = pd.DataFrame([dict(m, day=d, rat=ct.RATS[d.split("-")[0]]) for d, o in F3.items() for m in o["poke_post"]])
print(f"inbound rows (outer-poke-triggered, 5 s after the poke, arm-restricted): {len(PO)}; with a distance over a "
      f"drawn COM post-stopping range at >= 4 cm/s: {PO.d_post.notna().sum()}")


def f3d_table(com, inbound_col="d_post"):
    rows = []
    for rat in RATS:
        a = com[com.rat == rat].d_post.dropna()
        b = PO[PO.rat == rat][inbound_col].dropna()
        ok = len(a) > 0 and len(b) > 0
        rows.append({"rat": rat, "n_com": len(a), "n_inbound": len(b), "com_median_cm": a.median(), "inbound_median_cm": b.median(),
                     "p_greater": stats.ranksums(a, b, alternative="greater").pvalue if ok else np.nan,
                     "p_two_sided": stats.ranksums(a, b).pvalue if ok else np.nan})
    return pd.DataFrame(rows)


SP = pd.DataFrame([dict(m, day=d, rat=ct.RATS[d.split("-")[0]]) for d, o in F3.items() for m in o["com_seg_post"]])
f3d_readings = dict(pick)
f3d_readings["same measure: COM rows of the arm-restricted tables around COM time, within the arm (decoder A)"] = SP
f3d_readings["with the COM-aligned rows of the other tables, decoder A"] = pick_other["aligned rows added, decoder A"]
f3d_readings["with all rows of the other tables, decoder A"] = pick_other["all rows added, decoder A"]
f3d = {}
for name, d in f3d_readings.items():
    out = f3d_table(d)
    f3d[name] = out
    print(f"\nreading {name}:")
    print(out.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    print(f"rats with COM greater at p < 0.0001: {int((out.p_greater < 1e-4).sum())}; at p < 0.05: {int((out.p_greater < 0.05).sum())}")
print(f"\nrange of the distances: COM, arm-restricted {SP.d_post.min():.1f} to {SP.d_post.max():.1f} cm; inbound "
      f"{PO.d_post.min():.1f} to {PO.d_post.max():.1f} cm")


In [ ]:
# The inbound arm ranges are one random draw per row; the same test over 20 further draws, each with its own seed
draws, draw_p = [], []
for k in range(1, ct.N_DRAWS + 1):
    for name, d in f3d_readings.items():
        out = f3d_table(d, f"d_post_{k}")
        for rat, p_ in zip(out.rat, out.p_greater):
            draw_p.append({"draw": k, "reading": name.split(",")[0].split(":")[0], "rat": rat, "p": p_})
        draws.append({"draw": k, "reading": name.split(",")[0].split(":")[0], "rats_p_1e-4": int((out.p_greater < 1e-4).sum()),
                      "rats_p_0.05": int((out.p_greater < 0.05).sum()), "max_p_of_4th_best": float(np.sort(out.p_greater.values)[3])})
D3 = pd.DataFrame(draws)
print("rats with COM greater at p < 0.0001 over the 20 draws, per reading (columns: number of rats; values: number of draws):")
print(D3.groupby("reading")["rats_p_1e-4"].value_counts().unstack(fill_value=0).to_string())
print("\nthe fourth smallest p per draw, per reading: median [min, max]")
print(D3.groupby("reading").max_p_of_4th_best.describe()[["50%", "min", "max"]].to_string(float_format=lambda v: f"{v:.2g}"))
D3["full"], D3["partial"] = D3["rats_p_1e-4"] >= 4, D3["rats_p_0.05"] >= 4
print("\ndraws meeting the full bar (4 rats at p < 0.0001) and the partial bar (4 rats at p < 0.05), of 20:")
print(D3.groupby("reading")[["full", "partial"]].sum().to_string())
print("\nmedian p over the 20 draws, per rat and reading:")
print(pd.DataFrame(draw_p).groupby(["reading", "rat"]).p.median().unstack()[RATS].to_string(float_format=lambda v: f"{v:.2g}"))


**Result: not reproduced.** After stopping, the
largest maze distance on COM events is greater than on inbound runs over matched arm ranges
(one-sided rank-sum) with p < 0.0001 in 3 rats (M, K, L) and p < 0.05 in 4 (M, E, K, L) under the
smaller bound of the two decodes, and with p < 0.0001 in 4 rats (M, E, K, L) under the larger. The
outcome verified in round 2 rested on the smaller bound. The decoder can now be told: every arm-restricted
table, the inbound ones among them, has the decodes of one decoder (A), and the code takes the COM
and the control decodes of a figure from one decoder (figure4c.get_position_decode_data_day). Under
decoder A, p < 0.0001 in 3 rats and p < 0.05 in 3 (rat E has p = 0.053). The inbound arm ranges
are one random draw per row; over 20 further draws decoder A meets the partial bar (four rats at
p < 0.05) in 13 and the full bar in none, with a median p of 0.045 for E and 0.063 for J. Adding the
whole-track tables that match neither trigger, which have the windows of the COM tables (the rows
of the tables matching neither trigger lie within 5 s of this entry's COM times in 53 to 84% per
rat), decoder A gives p < 0.05 in 3 rats (E 0.09 with the rows within 20 ms of this entry's COM
times, 0.16 with all rows), and no further draw meets the partial bar under either. Measured within
the arm on both sides, from the arm-restricted COM tables, COM is greater at p < 0.05 in 3 rats (E
0.90, J 0.13), and 3 of 20 further draws meet the partial bar. So with the decoder identified and
all of the paper's COM tables, the claim is not reproduced; the third verification round agreed
with this change. The result is borderline: it rests on the first draw and on counting the tables
that match neither trigger as COM tables, which they may not be.
Only 296 of the 974 inbound rows have head positions at 4 cm/s or more within a drawn COM range.


## figs7b-no-swr

> The non-local activity during COM occurred during theta, not during sharp-wave ripples; COM
> periods had a lower ripple rate, while trial ends, rewarded or not, had higher rates.

SWRs are detected on the deposited 1 kHz LFP (`*_ecephys.nwb`, `scratch/filtered data`) as the
Methods (SWR detection) describe; the helper's docstring gives where each parameter comes from. Two
choices rest on the files: the electrode table gives no locations, so every tetrode in the LFP file
is used (the Methods include "all electrodes with visible SWRs"), and the corpus callosum reference
of the Methods is taken as each channel's `ref_elect_id` where that channel is in the file (reading
r1) or not subtracted (reading r0). The session baseline is estimated from 20 windows of 4 s spread
over each run epoch. Per rat, as many unrewarded and rewarded trial ends as COM trials are drawn
(sessions with at least 20 trials, fixed seed). The criterion's windows (a) and (b), and its model
readings (i) to (iii), are all run.


In [ ]:
# --- cell: build windows and run SWR detection
rng = np.random.default_rng(7)
Tq = T[T.nses >= ct.MIN_TRIALS]
EV = pd.concat([r["events"] for r in R.values() if len(r["events"])], ignore_index=True)
first = EV.sort_values("t_rev").groupby(["day", "epoch", "trial_start"])[["t_rev", "t_exit"]].first().reset_index()
cw = Tq[Tq.com].merge(first, left_on=["day", "epoch", "start"], right_on=["day", "epoch", "trial_start"])
rows = []
for x in cw.itertuples():
    rows.append(dict(rat=x.rat, day=x.day, epoch=x.epoch, kind="com", w0_a=x.t_rev - 1, w1_a=x.t_rev + 1,
                     w0_b=x.t_rev, w1_b=x.t_exit, read0=x.t_rev - 1, read1=max(x.t_rev + 1, x.t_exit)))
for rat, g in Tq.groupby("rat"):
    n = int((cw.rat == rat).sum())
    for rew, kind in ((False, "unrewarded"), (True, "rewarded")):
        pool = g[g.rewarded == rew]
        pick = pool.iloc[rng.choice(len(pool), size=min(n, len(pool)), replace=False)]
        for x in pick.itertuples():
            out = x.poke_out if np.isfinite(x.poke_out) else np.nan
            rows.append(dict(rat=rat, day=x.day, epoch=x.epoch, kind=kind, w0_a=x.end, w1_a=x.end + 2,
                             w0_b=x.end, w1_b=out, read0=x.end, read1=max(x.end + 2, out if np.isfinite(out) else 0)))
W = pd.DataFrame(rows)
print("windows per rat and kind:")
print(W.groupby(["rat", "kind"]).size().unstack()[["com", "unrewarded", "rewarded"]].loc[[r for r in RATS if r in set(W.rat)]].to_string())


In [ ]:
# --- cell: run per day
lfp_by_day = assets[assets.kind == "lfp"].set_index("day").url.to_dict()
swr_days = [d for d in sorted(W.day.unique()) if d in lfp_by_day]
jobs = [(lfp_by_day[d], assets[assets.day == d], R[d]["epochs"],
         W[W.day == d].drop(columns=["rat", "day", "kind"]).assign(i=W.index[W.day == d])) for d in swr_days]
t0 = time.time()
res = dict(zip(swr_days, ct.processes(ct.swr_job, jobs, workers=16)))
print(f"SWR detection on {len(res)} days in {time.time() - t0:.0f} s; days with windows but no LFP file: "
      f"{sorted(set(W.day) - set(lfp_by_day))}")


In [ ]:
# Check of the detection and the clocks: SWRs per second of immobility in the baseline windows,
# the LFP file's epochs against the raw file's, and the channels used.
BT = pd.concat([o[1].assign(day=d, rat=ct.RATS[d.split("-")[0]]) for d, o in res.items()], ignore_index=True)
imm = BT.groupby(["rat", "ref"])[["imm_events", "imm_s"]].sum()
imm["swr_per_immobile_s"] = imm.imm_events / imm.imm_s
print("SWRs per second of immobility (speed < 4 cm/s) in the baseline windows, per rat and reference reading:")
print(imm.round(3).to_string())
DET = pd.concat([o[2].assign(day=d, rat=ct.RATS[d.split("-")[0]]) for d, o in res.items()], ignore_index=True)
print("\ntetrodes used per day (min to max) and days with a reference channel in the file, per rat:")
print(DET.groupby(["rat", "day"]).agg(tetrodes=("ntrode", "size"), with_ref=("ref_col", lambda s: int((s >= 0).sum())))
      .groupby("rat").agg(tet_min=("tetrodes", "min"), tet_max=("tetrodes", "max"),
                          days=("tetrodes", "size"), days_with_ref=("with_ref", lambda s: int((s > 0).sum()))).to_string())
lfp_ep_ok = []
for d in swr_days[:3] + swr_days[-3:]:
    ep = ct.open_nwb(lfp_by_day[d])["intervals/epochs"]
    st = ep["start_time"][:]
    lfp_ep_ok.append((d, len(st), bool(np.allclose(np.sort(st), np.sort(R[d]["epochs"].start.values)))))
print("\nLFP file epochs equal to the raw file's (6 days):", lfp_ep_ok)


In [ ]:
# Clock check of the LFP: theta amplitude follows head speed at no shift, and not when the speed is shifted
shifts = [-30.0, -5.0, 0.0, 5.0, 30.0]
jobs_th = []
for d in swr_days[::3]:
    run_ep = R[d]["epochs"][R[d]["epochs"].tag.str.contains("Session")]
    sp = ct.speed_by_epoch(assets[assets.day == d], run_ep.iloc[:1])
    if sp:
        tag = list(sp)[0]
        jobs_th.append((lfp_by_day[d], sp[tag], float(run_ep.iloc[0].start), shifts))
th = pd.DataFrame(ct.processes(ct.theta_speed_check, jobs_th, workers=16), index=swr_days[::3][:len(jobs_th)])
print("Spearman correlation of theta amplitude with head speed shifted by s seconds (first run epoch, every third day):")
print(th.round(3).to_string())
print("median:", th.median().round(3).to_dict())


In [ ]:
# --- cell: rates and models
S = pd.concat([o[0].set_index("i") for o in res.values()]).join(W[["rat", "day", "kind"]])
for wn in ("a", "b"):
    S[f"dur_{wn}"] = S[f"w1_{wn}"] - S[f"w0_{wn}"]
    for rf in ("r0", "r1"):
        S[f"rate_{wn}_{rf}"] = S[f"n_{wn}_{rf}"] / S[f"dur_{wn}"]
print(f"windows: {len(W)}; with an LFP file: {len(S)}; with a baseline and counts: {S['n_a_r0'].notna().sum()}")
print("window lengths (s), reading b, median [min, max] by kind:")
print(S.groupby("kind").dur_b.describe()[["count", "50%", "min", "max"]].round(2).to_string())
KINDS = ["com", "unrewarded", "rewarded"]
rat_order = [r for r in RATS if r in set(S.rat)]
fits = []
for wn in ("a", "b"):
    for rf in ("r0", "r1"):
        col = f"rate_{wn}_{rf}"
        d = S[S[col].notna() & np.isfinite(S[col]) & (S[f"dur_{wn}"] > 0)].copy()
        m = d.groupby(["rat", "kind"])[col].mean().unstack()[KINDS].loc[rat_order]
        below = (m.com < m.unrewarded) & (m.com < m.rewarded)
        print(f"\nwindow {wn}, reference {rf}: mean ripple rate (events/s) per rat and kind; COM below both ends in "
              f"{int(below.sum())} of {len(m)} rats")
        print(m.assign(com_below_both=below).round(3).to_string())
        ind = pd.get_dummies(d.kind).astype(float)[KINDS]
        ratd = pd.get_dummies(d.rat).astype(float)[rat_order]
        y_raw = d[col]
        y_z = d.groupby("rat")[col].transform(lambda v: (v - v.mean()) / v.std() if v.std() > 0 else v * 0.0)
        variants = {
            "i, the three conditions and the rat indicators (first dropped), no constant": (y_raw, pd.concat([ratd.iloc[:, 1:], ind], axis=1)),
            "ii, z-scored within rat, the three conditions, no constant": (y_z, ind),
            "iii, constant + all rat indicators + the three conditions, pinv": (y_raw, pd.concat([pd.Series(1.0, index=d.index, name="const"), ratd, ind], axis=1)),
        }
        for vn, (y, X) in variants.items():
            f = sm.OLS(y.astype(float), X).fit()
            ci = f.conf_int()
            row = {"window": wn, "ref": rf, "reading": vn.split(",")[0], "n": len(d)}
            for k in KINDS:
                row[f"{k}_coef"] = f.params[k]
                row[f"{k}_lo"], row[f"{k}_hi"] = ci.loc[k, 0], ci.loc[k, 1]
                row[f"{k}_p"] = f.pvalues[k]
            row["holds"] = (row["com_coef"] < 0 and row["com_p"] < 0.05 and row["unrewarded_coef"] > 0 and
                            row["unrewarded_p"] < 0.05 and row["rewarded_coef"] > 0 and row["rewarded_p"] < 0.05)
            fits.append(row)
F7 = pd.DataFrame(fits)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
print("\nOLS fits (coefficient, 95% CI, p) per window, reference and reading:")
print(F7.to_string(index=False))


In [ ]:
# Reading (iii) with each rat left out in turn (verifier, round 2): rat K has no reference channel and
# the weakest theta-speed coupling, and with the reference the other rats' end rates are low.
def ols_pinv(y, X, rcond=1e-10):
    """OLS by pseudo-inverse, as statsmodels does it, with an explicit cutoff for the singular values so that
    the exact collinearity is always removed (statsmodels' default cutoff left it in some of these fits)."""
    Xv, yv = X.values.astype(float), y.values.astype(float)
    beta = np.linalg.pinv(Xv, rcond=rcond) @ yv
    rank = np.linalg.matrix_rank(Xv, tol=rcond * np.linalg.svd(Xv, compute_uv=False)[0])
    df = len(yv) - rank
    cov = (np.sum((yv - Xv @ beta) ** 2) / df) * np.linalg.pinv(Xv.T @ Xv, rcond=rcond)
    t = beta / np.sqrt(np.diag(cov))
    return pd.Series(beta, index=X.columns), pd.Series(2 * stats.t.sf(np.abs(t), df), index=X.columns)


rows = []
for wn in ("a", "b"):
    for rf in ("r0", "r1"):
        col = f"rate_{wn}_{rf}"
        d0 = S[S[col].notna() & np.isfinite(S[col]) & (S[f"dur_{wn}"] > 0)]
        for out in [None] + rat_order:
            d = d0 if out is None else d0[d0.rat != out]
            ind = pd.get_dummies(d.kind).astype(float)[KINDS]
            ratd = pd.get_dummies(d.rat).astype(float)[[r for r in rat_order if r != out]]
            params, pvalues = ols_pinv(d[col], pd.concat([pd.Series(1.0, index=d.index, name="const"), ratd, ind], axis=1))
            row = {"window": wn, "ref": rf, "left_out": out or "none", "n": len(d)}
            for k in KINDS:
                row[f"{k}_coef"], row[f"{k}_p"] = params[k], pvalues[k]
            row["holds"] = (row["com_coef"] < 0 and row["com_p"] < 0.05 and row["unrewarded_coef"] > 0 and
                            row["unrewarded_p"] < 0.05 and row["rewarded_coef"] > 0 and row["rewarded_p"] < 0.05)
            rows.append(row)
LOO = pd.DataFrame(rows)
big = (LOO[[f"{k}_coef" for k in KINDS]].abs() > 1e3).any(axis=1)
print(f"fits with a coefficient above 1000 in magnitude: {int(big.sum())}")
main = F7[F7.reading == "iii"].set_index(["window", "ref"])
none = LOO[LOO.left_out == "none"].set_index(["window", "ref"])
print("all-rat fits equal to the statsmodels fits of the cell above:",
      bool(np.allclose(none[[f"{k}_coef" for k in KINDS]].values, main[[f"{k}_coef" for k in KINDS]].values, rtol=1e-6)))
print("reading (iii), constant + rat indicators + the three conditions, with one rat left out:")
print(LOO.to_string(index=False))
print("fits that hold, per window and reference (of 6: all rats, then each rat left out):")
print(LOO.groupby(["window", "ref"]).holds.sum().to_string())


**Result: reproduced.** Under the criterion's model reading (iii), the fit with a constant, every
rat indicator and the three conditions by pseudo-inverse, the COM coefficient is negative and both
end-of-trial coefficients positive, each with p < 0.05, in both windows and with or without the
reference subtraction (window a, no subtraction: -0.043, 0.049 and 0.030 events/s, p from 3e-15 to
1e-6). Reading (ii), rates z-scored within rat, also holds without the subtraction (-0.22, 0.15 and
0.07 in window a), and reading (i) holds in none. The mean COM rate is below both end-of-trial
rates in all 5 rats in window a without the subtraction and in 4 of 5 in the other three
combinations. Against the paper's coefficients (arbitrary units, -0.18, 0.22 and 0.30), the
z-scored reading gives a COM coefficient of larger magnitude (-0.22) and smaller end-of-trial
coefficients (0.15 and 0.07, the rewarded one outside its tolerance). Ripples are rare everywhere
here: 0.13 to 0.34 per second of immobility in the baseline windows, and at most 0.22 per second at
trial ends. With one rat left out, reading (iii) holds in every fit without the subtraction; with
the subtraction it holds with any of M, E, J or L left out and fails without rat K (window a, COM
-0.0009, p = 0.62; window b, unrewarded end -0.011), so with the subtraction the result rests on rat
K, which has no reference channel in its files. The outcome rests on three choices the files force
(all tetrodes of the LFP file, the reference channel, a sampled baseline), stated above. The LFP
shares the behavioral clock: theta amplitude follows head speed at no shift (median Spearman 0.68)
and not at shifts of 5 or 30 s (median -0.20 to -0.02), though weakly in rat K (0.13 to 0.18, and
-0.06 on one day).


## Blocked claims

The other claims on hippocampal content (Figures 4, 5, S3, S6, S8, S10, S11) need the clusterless
state-space decoder's posterior per 2 ms bin, its continuous-state probability, and the remote and
local extended events built on it, over every COM event. None of these is in the dandiset: the
triggered tables hold only the MAP within windows around COM times and outer pokes, without the
posterior or the state probabilities. The decoder would have to be refit from the 30 kHz voltage
(12.5 TB of raw files, 313 GB per day on average) after spike detection, which does not fit an
entry's time limit. The cell below gives the sizes.


In [ ]:
print("triggered tables per rat:", kinds.groupby("rat").triggered.sum().to_dict())
lfp_assets = assets[assets.kind == "lfp"]
print(f"LFP files: {len(lfp_assets)}, {lfp_assets['size'].sum() / 1e9:.0f} GB; per rat {lfp_assets.groupby('rat').size().to_dict()}")
f = ct.open_nwb(lfp_assets.iloc[0].url)
es = f["scratch/filtered data"]
t0 = time.time()
block = es["data"][:600000, :]
dt = time.time() - t0
print(f"{lfp_assets.iloc[0].path}: {es['data'].shape} samples x channels, described as '{es.attrs['description']}'; "
      f"reading 600 s took {dt:.1f} s ({block.nbytes / 1e6 / dt:.0f} MB/s)")
raw_assets = assets[assets.kind == "raw"]
print(f"raw files: {raw_assets['size'].sum() / 1e12:.1f} TB, {raw_assets['size'].mean() / 1e9:.0f} GB per day on average")
pos = assets[assets.kind == "obj_behavior"].iloc[0]
ho = ct.open_nwb(pos.url)["scratch/CompassDirection/head_orientation/data"]
print(f"head orientation in {pos.path}: unit '{ho.attrs['unit']}', range {np.nanmin(ho[:]):.2f} to {np.nanmax(ho[:]):.2f}")
print(f"elapsed {time.time() - started:.0f} s")

## Summary

| Claim | Outcome |
|---|---|
| fig1c-learning | reproduced |
| fig1e-trial-counts | not reproduced |
| fig1d-initial-switch-bias | reproduced |
| fig1d-switch-bias-declines | partially reproduced |
| figs1f-multiple-com | reproduced |
| fig1g-com-corrective | reproduced |
| fig1g-com-matches-run-through | reproduced |
| fig1h-final-choice-less-switching | not reproduced |
| fig1h-final-ratio-near-one | reproduced |
| fig3b-extended-before-stopping | partially reproduced |
| fig3b-distance-predicts-com | partially reproduced |
| fig3c-distant-after-stopping | partially reproduced |
| fig3d-distant-vs-inbound | not reproduced |
| figs7b-no-swr | reproduced |
| the other 15 claims on hippocampal content | blocked, too large |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. Trials are built from the digital poke and pump
events, and agree with the Statescript logs to within one trial per session. COM trials are
detected on the deposited 500 Hz linearization with the Methods' rule; both thresholds of the
claims file are run (10 cm and 0.1 of the 96 cm arm) and give the same outcomes for every claim
but the counts. The closing structure of the reversal detection is a standard choice. The per-rat
sequence is read from the reward log. Each rat's peak day is its last day in the dandiset. The
Figure 3 claims are tested on the paper's own triggered tables, identified by their trigger times.
The whole-track tables come in pairs from two decoders; the files do not name them, but one (A)
made every arm-restricted table, and the code takes a figure's COM and control decodes from one
decoder, so A is assumed for fig3d. The control arm ranges are a random draw, as in the code, and
the Figure 3B and 3D results are shown over 20 further draws; fig3b-extended and fig3d sit at the
edge of a bar and change with the draw. SWRs are detected on the deposited LFP with the Methods' parameters, using every
tetrode in the LFP file (no locations are given), with and without subtracting the reference
channel, and a session baseline sampled from 20 windows per epoch; rat M's fifth day has no LFP
file, so its 64 windows are dropped.

What was found about the dataset. The dandiset holds fewer trials than the paper reports for
rats M (5 days against 6 in the paper), K and L. Rat M's 500 Hz linearizations are each deposited
twice, as identical copies, its Statescript logs are named "syayescript", and it has 4 LFP files
for 5 days. None of the Spyglass analysis files (2415 of 2494 assets) says what it holds; the
triggered tables can be read only with the paper's code and an alignment to events, and the
parameters that made each (trigger, window, decoder) are not in the files. The electrode table has
no locations. The decoder's posterior and the events the other neural claims rest on are not
deposited. The reward log gives the sequence 3-4-2-1-3 for M, E and J and 1-2-4-3-1 for K and L,
against Figure S1E's labels.